# Part 12 · Notebook 07 — Deployment, CI and backtest regression tests

**Sessions:** S13–S14 (Deployment with Docker Compose · CI/CD & the release process) · [Lesson plan](../../docs/lessons/PART_12_TRADING_PLATFORM_CAPSTONE.md) · graded labs in [`labs/part12/`](../../labs/part12/)

**You will:**
1. Check a docker-compose file against production rules: pinned images, no secrets inline, restart policies, a database health check, one exposed port.
2. Check that the CI workflow runs every required gate.
3. Freeze a backtest's results as a regression baseline, and watch it catch bugs that look like improvements.

How these notebooks work: the setup, data and plotting code is written for you. Cells marked **✍️ Your turn** need a few lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.
Everything runs offline on synthetic data with known answers (planted violations, injected faults, a strategy that stops working): no broker, Docker or network is needed.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p12lib.py is in notebooks/part12/
    sys.path.insert(0, str(d))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p12lib as p

p.use_course_style()

## 1. The compose file is code, so review it with code

The paper stack runs one container per concern: IB Gateway, the engine, the API, PostgreSQL/TimescaleDB, Redis, n8n, Prometheus, Grafana, and a reverse proxy in front. `p.check_compose` encodes the S13 rules: images pinned to an exact version (never `latest`), secrets only through env files or `${...}` references, restart policies on the critical services, a health check on the database and dependents that wait for it, and no published ports except the proxy's. The committed `deploy/docker-compose.yml` passes. The lesson plan's teaching version does not, and neither does a 2 a.m. "quick fix":

In [ ]:
import copy
import yaml

LESSON_PLAN_COMPOSE = """
services:
  ib-gateway:
    image: ${IB_GATEWAY_IMAGE}
    env_file: ./secrets/ib.env
    environment: {TRADING_MODE: paper}
    restart: unless-stopped
  db:
    image: timescale/timescaledb:latest-pg16
    volumes: [dbdata:/var/lib/postgresql/data]
    env_file: ./secrets/db.env
    healthcheck: {test: ["CMD-SHELL", "pg_isready -U quantforge"], interval: 10s, retries: 5}
  redis:
    image: redis:7
  engine:
    build: ..
    env_file: ./secrets/engine.env
    depends_on:
      db: {condition: service_healthy}
    restart: unless-stopped
  api:
    build: ..
    env_file: ./secrets/engine.env
    depends_on: [db]
  n8n:
    image: n8nio/n8n
  prometheus:
    image: prom/prometheus
  grafana:
    image: grafana/grafana
"""

committed = yaml.safe_load((p.DEPLOY_DIR / "docker-compose.yml").read_text())
print("deploy/docker-compose.yml:", p.check_compose(committed) or "passes")
print("\nthe lesson plan's teaching version:")
for problem in p.check_compose(yaml.safe_load(LESSON_PLAN_COMPOSE)):
    print("   ", problem)
hotfix = copy.deepcopy(committed)
hotfix["services"]["db"]["environment"] = {"POSTGRES_PASSWORD": "changeme123"}   # "just to get it running"
hotfix["services"]["db"]["ports"] = ["5432:5432"]                                  # "so I can connect from my laptop"
hotfix["services"]["engine"]["image"] = "quantforge:latest"
print("\nthe quick fix:", p.check_compose(hotfix))

Each finding is an incident waiting to happen: an unpinned image changes under you on the next pull, an inline password ends up in Git, a database port open to the internet gets scanned within hours.

## 2. The CI gates

The S14 pipeline must run lint (`ruff check`), strict types (`mypy --strict`), the architecture rules (`lint-imports`), tests with a coverage floor (`--cov-fail-under`) and the backtest regression tests (`tests/regression`). `p.REQUIRED_CI` maps each gate to the text that proves it runs. Collect the `run` command of every step of every job, and report the gates whose text is missing.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def check_ci(workflow):
    runs = ...                                                       # ✍️ every step's "run" (default "") of every job, joined into one string
    return sorted(k for k, needle in p.REQUIRED_CI.items() if needle not in runs)

ci_text = (p.DEPLOY_DIR / "ci.yml").read_text()
ci = yaml.safe_load(ci_text)
no_regression = copy.deepcopy(ci)
no_regression["jobs"]["test"]["steps"] = [s for s in ci["jobs"]["test"]["steps"] if "regression" not in s.get("run", "")]
loosened = yaml.safe_load(ci_text.replace("mypy --strict", "mypy").replace(" --cov-fail-under=85", ""))
cases = {"committed ci.yml": ci, "regression step deleted": no_regression, "loosened": loosened}
mine = p.check("check_ci", {k: p.attempt(check_ci, w) for k, w in cases.items()}, {k: p.check_ci(w) for k, w in cases.items()})
mine

Gates erode quietly: someone drops `--strict` to get a release out, and nobody notices until the type errors pile up. A check like this can run on the workflow file itself.

## 3. Backtest regression tests

Unit tests check small pieces; a **regression test** checks that behaviour didn't change: a frozen dataset and config must reproduce the stored results. Here the fixture is an SMA(10/40) crossover on `p.ohlcv()`, and the baseline is its total return, number of trades and exposure. Then five changes to the SMA: a harmless refactor and four bugs.

In [ ]:
import json

close = p.ohlcv()["close"].to_numpy()                               # the frozen fixture dataset
baseline = p.fixture_backtest(close)
print("stored baseline (tests/regression/baseline.json):", json.dumps(baseline))

def sma_cumsum(x, n):                                                # the same maths, computed from a running sum
    c = np.r_[0.0, np.cumsum(x)]
    return np.r_[np.full(n - 1, np.nan), (c[n:] - c[:-n]) / n]

variants = {"refactor: SMA from a running sum": sma_cumsum,
            "bug: window n − 1": lambda x, n: p.sma(x, n - 1),
            "bug: rolling(min_periods=1)": lambda x, n: pd.Series(x).rolling(n, min_periods=1).mean().to_numpy(),
            "bug: one-bar look-ahead": lambda x, n: np.r_[p.sma(x, n)[1:], np.nan]}
results = {name: p.fixture_backtest(close, sma_fn=f) for name, f in variants.items()}
results["bug: a key dropped from the report"] = {k: v for k, v in baseline.items() if k != "exposure"}

Compare a result with the baseline: list (sorted) every key that is missing on either side, or whose values differ beyond `np.isclose(result, baseline, rtol=rtol, atol=1e-12)`. An empty list means behaviour is unchanged.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def regression_check(result, baseline, rtol=1e-9):
    bad = []
    for k in sorted(set(result) | set(baseline)):
        if ...:                                                      # ✍️ missing on either side, or not close
            bad.append(k)
    return bad

mine = p.check("regression_check", {k: p.attempt(regression_check, r, baseline) for k, r in results.items()},
               {k: p.regression_check(r, baseline) for k, r in results.items()})
pd.DataFrame(results).T.assign(changed=[", ".join(v) or "—" for v in mine.values()])

The refactor passes: its differences are floating-point noise. Every bug is caught, and the dangerous one is the look-ahead: it *raises* the total return and keeps the same number of trades, so in a code review it looks like an improvement. A changed baseline is never re-frozen silently: the pull request must explain why the numbers moved.

**Release process:** semantic versions and a changelog; deploy to paper first; promote to live only after a paper soak; feature flags to switch a strategy off without a deploy; a written rollback procedure (the previous image tag is one line in the compose file, which is one more reason to pin it).

## Wrap-up

* The deployment and the pipeline are files, so they can be checked automatically.
* Pinned images, secrets outside Git, restart policies, health checks, one exposed port.
* Regression baselines catch behaviour changes that unit tests and reviewers miss.
* Graded version: `labs/part12/week44_production` (`check_compose`, `check_ci`, `fixture_backtest`, `regression_check`).